[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/n8n-workflows/01-introduccion-n8n.ipynb)

# Introducción a n8n con Claude

Este notebook simula en Python los patrones de n8n para que puedas entender
la lógica antes de implementarlos en el orquestador visual.

**Lo que aprenderás:**
- Patrón Email → Claude → Acción (clasificación y enrutamiento)
- Función reutilizable `llamar_claude()` equivalente al helper de n8n
- Manejo básico de errores y workflow de errores centralizado

In [ ]:
import anthropic
import json
import os
from datetime import datetime

client = anthropic.Anthropic()
print("Cliente Anthropic listo")

## Función reutilizable: llamar_claude()

Equivalente al helper JavaScript de n8n que se pega en cualquier Code node.

In [ ]:
def llamar_claude(prompt: str, modelo: str = "claude-haiku-4-5-20251001", 
                  max_tokens: int = 500, system: str = None) -> str:
    """Equivalente a la función llamarClaude() de los Code nodes en n8n."""
    kwargs = {
        "model": modelo,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": prompt}]
    }
    if system:
        kwargs["system"] = system
    
    response = client.messages.create(**kwargs)
    return response.content[0].text

# Prueba básica
resultado = llamar_claude("Di 'Hola n8n' en una sola frase corta.")
print(resultado)

## Workflow 1: Email → Claude → Slack

### Nodo 1: Simular Gmail Trigger

En n8n el Gmail Trigger emite un objeto JSON por cada email recibido.

In [ ]:
# Simula la salida del nodo Gmail Trigger
EMAILS_EJEMPLO = [
    {
        "from": "cliente@empresa.com",
        "subject": "Urgente: fallo en producción",
        "snippet": "El sistema está caído desde hace 30 minutos. Estamos perdiendo ventas.",
        "id": "msg_001"
    },
    {
        "from": "ventas@proveedor.com",
        "subject": "Oferta especial Q2 2026",
        "snippet": "Te presentamos nuestra nueva oferta de licencias para equipos grandes...",
        "id": "msg_002"
    },
    {
        "from": "maria.garcia@cliente.es",
        "subject": "Re: Factura marzo 2026",
        "snippet": "Hola, ¿podéis reenviar la factura de marzo? No la encuentro en el email anterior.",
        "id": "msg_003"
    }
]

print(f"Simulando {len(EMAILS_EJEMPLO)} emails recibidos")

In [ ]:
# Nodo 2: HTTP Request → Claude API (clasificar email)
def clasificar_email(email: dict) -> dict:
    """Equivalente al nodo HTTP Request + Code Node en n8n."""
    prompt = f"""Clasifica este email. Responde SOLO con JSON válido, sin markdown.

{{"categoria": "soporte_urgente|soporte_normal|comercial|facturacion|spam",
 "prioridad": "alta|media|baja",
 "resumen": "1 frase",
 "es_urgente": true/false}}

Asunto: {email['subject']}
Mensaje: {email['snippet']}"""

    respuesta = llamar_claude(prompt, max_tokens=200)
    clasificacion = json.loads(respuesta)
    
    # Enriquecer con datos del email (equivalente al Code Node de parseo)
    clasificacion["from"] = email["from"]
    clasificacion["subject"] = email["subject"]
    clasificacion["email_id"] = email["id"]
    
    return clasificacion

# Procesar el primer email
resultado = clasificar_email(EMAILS_EJEMPLO[0])
print(json.dumps(resultado, indent=2, ensure_ascii=False))

In [ ]:
# Nodo 3: IF → Switch → Slack (enrutamiento)
def enrutar_email(clasificacion: dict) -> dict:
    """Equivalente al nodo IF + Switch en n8n."""
    canal_slack = {
        "soporte_urgente": "#soporte-urgente",
        "soporte_normal": "#soporte-general",
        "comercial": "#ventas",
        "facturacion": "#contabilidad",
        "spam": None  # No enviar
    }.get(clasificacion["categoria"], "#general")
    
    if canal_slack is None:
        return {"accion": "ignorar", "motivo": "spam"}
    
    # Simular envío a Slack (en n8n sería el nodo Slack postMessage)
    emoji = "🚨" if clasificacion["es_urgente"] else "📧"
    mensaje_slack = f"{emoji} Email de {clasificacion['from']}\n*{clasificacion['subject']}*\n{clasificacion['resumen']}"
    
    print(f"[SLACK → {canal_slack}]\n{mensaje_slack}\n")
    
    return {
        "accion": "enviado",
        "canal": canal_slack,
        "mensaje": mensaje_slack
    }

# Ejecutar workflow completo para todos los emails
print("=" * 50)
print("EJECUTANDO WORKFLOW: Email → Claude → Slack")
print("=" * 50)

for email in EMAILS_EJEMPLO:
    print(f"\nProcesando: {email['subject']}")
    clasificacion = clasificar_email(email)
    resultado = enrutar_email(clasificacion)
    print(f"Categoría: {clasificacion['categoria']} | Prioridad: {clasificacion['prioridad']}")

## Manejo de errores: workflow centralizado

En n8n, cuando un nodo falla se activa el Error Workflow. Aquí simulamos ese patrón.

In [ ]:
import traceback

ERRORES_LOG = []  # En n8n esto se guardaría en Google Sheets

def workflow_errores(workflow_nombre: str, nodo: str, error: Exception, execution_id: str):
    """Simula el Error Workflow de n8n: loguea y notifica."""
    registro = {
        "workflow": workflow_nombre,
        "nodo": nodo,
        "error": str(error),
        "timestamp": datetime.now().isoformat(),
        "execution_id": execution_id
    }
    ERRORES_LOG.append(registro)
    
    # En n8n: HTTP Request a Slack webhook + append a Google Sheets
    print(f"❌ Error en workflow '{workflow_nombre}'")
    print(f"   Nodo: {nodo}")
    print(f"   Error: {error}")
    print(f"   Guardado en log de errores.")
    
    return registro

def ejecutar_workflow_con_errores(email: dict) -> dict:
    """Wrapper que captura errores como hace n8n con el Error Workflow."""
    try:
        clasificacion = clasificar_email(email)
        return enrutar_email(clasificacion)
    except json.JSONDecodeError as e:
        return workflow_errores("Email-Claude-Slack", "Code: parsear JSON", e, "exec_001")
    except Exception as e:
        return workflow_errores("Email-Claude-Slack", "HTTP Request Claude", e, "exec_001")

# Simular un email con contenido que podría causar problemas
email_problematico = {
    "from": "test@test.com",
    "subject": "Test",
    "snippet": "Mensaje normal",
    "id": "msg_test"
}

resultado = ejecutar_workflow_con_errores(email_problematico)
print(f"\nResultado: {resultado}")
print(f"\nLog de errores acumulados: {len(ERRORES_LOG)}")

## Expresiones de n8n: equivalentes en Python

n8n usa `{{ $json.campo }}` en la interfaz visual. Aquí vemos el equivalente Python.

In [ ]:
# En n8n las expresiones acceden a datos de nodos anteriores:
# {{ $json.subject }}                      → email["subject"]
# {{ $('Gmail Trigger').first().json.from }} → email["from"]
# {{ $env.ANTHROPIC_API_KEY }}             → os.environ["ANTHROPIC_API_KEY"]
# {{ new Date().toISOString() }}           → datetime.now().isoformat()

# Ejemplo de construcción de prompt con expresiones (como en n8n):
email = EMAILS_EJEMPLO[0]

# En n8n esto se escribe en el campo body del nodo HTTP Request:
# content: "Clasifica: {{ $json.subject }}\n{{ $json.snippet }}"
prompt_con_expresiones = f"Clasifica: {email['subject']}\n{email['snippet']}"

# Resultado del nodo HTTP Request se accede en el Code Node siguiente como:
# $input.first().json.body.content[0].text  → en Python: response.content[0].text

print("Equivalencias n8n ↔ Python:")
print(f"  $json.subject → {email['subject']}")
print(f"  $env.ANTHROPIC_API_KEY → {os.environ.get('ANTHROPIC_API_KEY', 'sk-ant-...')[:15]}...")
print(f"  new Date().toISOString() → {datetime.now().isoformat()}")

## Resumen

| Concepto n8n | Equivalente Python | Notas |
|---|---|---|
| Gmail Trigger | `dict` con datos del email | n8n escucha eventos reales |
| HTTP Request → Claude | `client.messages.create()` | n8n usa nodo visual |
| Code Node | función Python | Mismo JavaScript en n8n |
| IF Node | `if/else` | n8n bifurca el flujo visualmente |
| Slack Node | `print()` simulado | n8n usa credenciales de Slack |
| `$json.campo` | `dict["campo"]` | Sintaxis de expresiones n8n |
| `$env.VAR` | `os.environ["VAR"]` | Variables de entorno |
| Error Workflow | `try/except` | n8n lo configura por workflow |

**Siguiente paso:** Implementa este mismo flujo en n8n arrastrando los nodos y conectándolos.
El Code Node acepta exactamente el mismo JavaScript del tutorial.